# Feature PDM

### Imports

In [ ]:

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


from ppvm_py.utils import load_object

from ppvm_py.plotting import plot_timeseries
from ppvm_py.plotting import plot_heatmap
from ppvm_py.plotting import generate_2d_video

from ppvm_py.data_processing.patt_fld8v import print_xyz_slice_example_from_parsed_data

from ppvm_py.data_processing.utils import create_index_to_coord_map

from ppvm_py.feature_generation.sdc import compute_symmetric_difference_coefficients_with_index_to_coord_map

from ppvm_py.data_utils_3d import find_closest_coordinate_index

### General Args

In [2]:
# IMPORTANT: Restart Kernel when swapping datasets to reload them!
Ha = 300

if Ha == 300:
    patt_fld8v_folder_path = Path("../../../../../data/4pi_re1000_ha300.360_ch28.p1/patt_fld8v/")
elif Ha == 1000:
    patt_fld8v_folder_path = Path("../../../../../data/4pi_re1000_ha1000.384.pi/patt_fld8v/")

fld_path = patt_fld8v_folder_path / Path("pkl/patt_fld8v.pkl")

# Estimates from evaluations
sigma = -1  # According to the results here.
B_z = -1  # According to the results here.

animation_folder = Path(f"../output/animations/feature_pdm/")

truncate_data = False
truncate_at_snapshot = 10  # Including indices 0 to (truncate_at_snapshot - 1)

test_size = 0.3

sdc_applicable_region = (
    slice(None),  # time
    slice(None),  # x
    slice(1, -1),  # y, sdc in y => boundaries in y are not applicable
    slice(None),  # z
)

##### Coordinates

In [ ]:
# SELECT ONE:
# 1: (1.5, 0, 0)
# 2: (8, 0, 0)
# ...
# (x, y, z) in [0, 12.56637] x [-1, 1] x [-1, 1]
location = 1

# When closest coordinates are not determined yet.
# Note: if 'True' is set for the first time, kernel must be restarted.
find_closest_coordinates = False

In [ ]:
if location == 1:
    x_coord_desired = 1.5
    y_coord_desired = 0
    z_coord_desired = 0
    # closest available grid locations for desired ones have to be determined further below after data loading

    # Adjust according to error message. No error will be thrown after correct declaration.
    if Ha == 300:
        x_coord = 1.489348
        y_coord = 3.161188e-17
        z_coord = 3.161188e-17
        i_x = 64
        i_y = 45
        i_z = 45
    elif Ha == 1000:
        x_coord = np.nan
        y_coord = np.nan
        z_coord = np.nan
        i_x = np.nan
        i_y = np.nan
        i_z = np.nan

elif location == 2:
    x_coord_desired = 8
    y_coord_desired = 0
    z_coord_desired = 0

    # Adjust according to error message. No error will be thrown after correct declaration.
    if Ha == 300:
        x_coord = 8.005244
        y_coord = 3.161188e-17
        z_coord = 3.161188e-17
        i_x = 344
        i_y = 45
        i_z = 45
    elif Ha == 1000:
        x_coord = np.nan
        y_coord = np.nan
        z_coord = np.nan
        i_x = np.nan
        i_y = np.nan
        i_z = np.nan

### Preamble

In [5]:
mean_mse = {}  # Dict for gathering mean-squared-errors.

### Load Data

In [6]:
if 'fld_data' not in locals():
    fld_data = load_object(fld_path)
    if truncate_data:
        fld_data['timeseries'] = fld_data['timeseries'][:truncate_at_snapshot]
    fld_data['index_to_coord_map'] = create_index_to_coord_map(fld_data['coord_to_index_map'])
    if not find_closest_coordinates:
        fld_data['timeseries'] = fld_data['timeseries'][
            :,
            i_x:i_x + 1,
            i_y - 1:i_y + 2,  # for sdc
            i_z:i_z + 1,
            :,
        ]
        fld_data['coord_to_index_map'] = {
            (
                fld_data['index_to_coord_map'][
                    i_x,
                    i_y,
                    i_z,
                ][0],
                fld_data['index_to_coord_map'][
                    i_x,
                    i_y - 1 + j,
                    i_z,
                ][1],
                fld_data['index_to_coord_map'][
                    i_x,
                    i_y,
                    i_z,
                ][2],
            ): (
                0,
                j,
                0,
            )
            for j in range(3)
        }
    fld_data['index_to_coord_map'] = create_index_to_coord_map(fld_data['coord_to_index_map'])
print_xyz_slice_example_from_parsed_data(fld_data)

N, n_x, n_y, n_z, n_v = fld_data['timeseries'].shape  # N = num snapshots

Object successfully loaded from C:\Users\hydro\Documents\PhD\nuclear_fusion_cooling\data\4pi_re1000_ha300.360_ch28.p1\patt_fld8v\pkl\patt_fld8v.pkl
The 3D data looks as follows:
N (number of snapshots) = 100
n_x (grid depth) = 1, n_y (grid height) = 3, n_z (grid width) = 1, n_v (number of variables) = 8
Labels:
['vx', 'vy', 'vz', 'jx', 'jy', 'jz', 'P', 'F']
Preview of the first snapshot (first 2 x, first 2 y, first 2 z) with coordinates:
  (x=1.49, y=-0.02, z=0.00): [ 2.035808e+01  8.793886e-01 -8.495488e+00 -8.699981e+00 -4.448584e+00
 -3.472447e+00 -4.837785e-04 -6.533117e+00]
  (x=1.49, y=0.00, z=0.00): [ 2.291906e+01 -7.508148e-01 -1.288873e+01 -9.818085e+00 -6.305217e+00
 -3.241020e+00  7.128800e-05 -6.821968e+00]


### Get Existing x, y and z Coordinates

In [7]:
# Fot initial determination of closest coordinates.
if find_closest_coordinates:
    x_coord = x_coord_desired
    y_coord = y_coord_desired
    z_coord = z_coord_desired
i_x, i_y, i_z = find_closest_coordinate_index(
    fld_data['coord_to_index_map'],
    x_coord=x_coord,
    y_coord=y_coord,
    z_coord=z_coord,
)

### Define Train and Test Set

In [8]:
fld_data_train, fld_data_test = train_test_split(
    fld_data['timeseries'],
    test_size=test_size,
    shuffle=False,
)
n_timesteps_train = fld_data_train.shape[0]
n_timesteps_test = fld_data_test.shape[0]

n_grid_points_no_y_boundary = n_x * (n_y - 2) * n_z

### Define $y_{true}$

In [9]:
y_true_train = fld_data_train[
    ...,
    fld_data['labels'].index('vx'),
][sdc_applicable_region]
y_true_train = y_true_train.reshape(n_timesteps_train, -1)

y_true_test = fld_data_test[
    ...,
    fld_data['labels'].index('vx'),
][sdc_applicable_region]
y_true_test = y_true_test.reshape(n_timesteps_test, -1)


# Every grid point is a feature. The scaling must not be different between these features => train SS on all data together.
ss_y_true_trainer = StandardScaler()
ss_y_true_trainer.fit(y_true_train.reshape(-1, 1))

# Create on SS that works on all features separately with the training params from the SS on all data.
ss_y_true = StandardScaler()

ss_y_true.scale_ = np.full(
    n_grid_points_no_y_boundary,
    ss_y_true_trainer.scale_[0],
)
ss_y_true.mean_ = np.full(
    n_grid_points_no_y_boundary,
    ss_y_true_trainer.mean_[0],
)
ss_y_true.var_ = np.full(
    n_grid_points_no_y_boundary,
    ss_y_true_trainer.var_[0],
)
ss_y_true.n_features_in_ = n_grid_points_no_y_boundary
ss_y_true.n_samples_seen_ = np.full(
    n_grid_points_no_y_boundary,
    ss_y_true_trainer.n_samples_seen_,
)


# trans y_true
y_true_train = ss_y_true.transform(
    y_true_train,
    copy=False,
    )
y_true_test = ss_y_true.transform(
    y_true_test,
    copy=False,
    )

### Precompute Features

In [10]:
approx_part_phi_y_train= compute_symmetric_difference_coefficients_with_index_to_coord_map(
    fld_data_train[..., fld_data['labels'].index('F')],
    direction_axis=2,
    index_to_coord_map=fld_data['index_to_coord_map'],
)
approx_part_phi_y_train = approx_part_phi_y_train.reshape(n_timesteps_train, -1)

approx_part_phi_y_test= compute_symmetric_difference_coefficients_with_index_to_coord_map(
    fld_data_test[..., fld_data['labels'].index('F')],
    direction_axis=2,
    index_to_coord_map=fld_data['index_to_coord_map'],
)
approx_part_phi_y_test = approx_part_phi_y_test.reshape(n_timesteps_test, -1)


v_x_pdm_train = approx_part_phi_y_train / B_z
v_x_pdm_train = ss_y_true.transform(
    v_x_pdm_train,
    copy=False,
)

v_x_pdm_test = approx_part_phi_y_test / B_z
v_x_pdm_test = ss_y_true.transform(
    v_x_pdm_test,
    copy=False,
)

j_y_test = fld_data_test[..., fld_data['labels'].index('jy')][sdc_applicable_region]
j_y_test = j_y_test.reshape(n_timesteps_test, -1)

### Potential Difference Method (SDC) Benchmark
Assume we know $j_y$, but we are restricted the the potential measurements $\phi$ from our DNS.

A simple feature used in the experiments is the symmetric difference coefficient $\frac{\Delta{\phi}}{\Delta{y}}$ to approximate $\frac{\partial{\phi}}{\partial{y}}$. This is also called *Potential Difference Method*. With this and $j_y$, Ohm's law can be approximated to calculate $v_x$.

We use the smallest difference possible to calculate $\frac{\Delta{\phi}}{\Delta{y}}$. With this, we have a benchmark for methods that use the Symmetric Difference Coefficient as singular feature.

##### Testing

In [11]:
v_x_sdc_test = (approx_part_phi_y_test - (j_y_test / sigma)) / B_z
y_pred_sdc_test = ss_y_true.transform(
    v_x_sdc_test,
    copy=False,
)

In [12]:
mse_sdc = mean_squared_error(
    y_true_test,
    y_pred_sdc_test,
    multioutput='raw_values',
)

mean_mse_sdc = np.mean(mse_sdc)
mean_mse['PDM Benchmark'] = float(mean_mse_sdc)
print(f"mean(mse(pdm_bench)) = {mean_mse_sdc}")

mean(mse(pdm_bench)) = 0.038441716089442066


### Potential Difference Method (PDM)

##### Testing

In [13]:
mse_pdm = mean_squared_error(
    y_true_test,
    v_x_pdm_test,
    multioutput='raw_values',
    )

mean_mse_pdm = np.mean(mse_pdm)
mean_mse['PDM'] = float(mean_mse_pdm)
print(f"mean(mse(pdm)) = {mean_mse_pdm}")

mean(mse(pdm)) = 1.100974245115424


### Linearly Scaled PDM (cPDM)

##### Training

In [14]:
lr_c_pdm = LinearRegression(fit_intercept=False)
lr_c_pdm.fit(
    X = v_x_pdm_train.reshape(-1, 1),
    y = y_true_train.reshape(-1, 1),
)
c =  1 / lr_c_pdm.coef_[0, 0]

##### Testing

In [15]:
y_pred_c_pdm_test = v_x_pdm_test / c

mse_c_pdm = mean_squared_error(
    y_true_test,
    y_pred_c_pdm_test,
    multioutput='raw_values',
)
mean_mse_c_pdm = np.mean(mse_c_pdm)
mean_mse['c-PDM'] = float(mean_mse_c_pdm)
print(f"mean(mse(c-PDM)) = {mean_mse_c_pdm}")

mean(mse(c-PDM)) = 0.743141199242183


### Linear Regression on Feature: PDM

##### Training

In [16]:
lr_pdm = LinearRegression()
lr_pdm.fit(
    X = v_x_pdm_train.reshape(-1, 1),
    y = y_true_train.reshape(-1, 1),
);

##### Testing

In [17]:
y_pred_lr_pdm_test = lr_pdm.predict(
    v_x_pdm_test.reshape(-1, 1),
)
y_pred_lr_pdm_test = y_pred_lr_pdm_test.reshape(
    n_timesteps_test,
    n_grid_points_no_y_boundary,
)

mse_lr_pdm = mean_squared_error(
    y_true_test,
    y_pred_lr_pdm_test,
    multioutput='raw_values',
)
mean_mse_lr_pdm = np.mean(mse_lr_pdm)
mean_mse['LR-PDM'] = float(mean_mse_lr_pdm)
print(f"mse(LR-PDM) = {mean_mse_lr_pdm}")

mse(LR-PDM) = 0.455671926486147


### Comparison

In [18]:
mean_mse

{'PDM Benchmark': 0.038441716089442066,
 'PDM': 1.100974245115424,
 'c-PDM': 0.743141199242183,
 'LR-PDM': 0.455671926486147}

### Appendix

##### Visualize j_y

In [19]:
j_y_train = fld_data_train[..., fld_data['labels'].index('jy')]
j_y_mean_in_time = np.mean(
    j_y_train,
    axis=0,
)
j_y_mean_in_time

array([[[-15.43801274],
        [-15.35482612],
        [-15.55513066]]])